In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

# from src.models.update import Update, neighbours
# # from src.models.graph_inference import Graph as Graph_interference
# from src.models.graph_training import Graph as Graph_train
# from src.models.bundle_adjustment import BundleAdjustment
from src.models.dpso_training import DPSO_train as DPSO

from src.data_loader.data_module_lightning import SonarSimDataModule

from src.training.metrics import pose_err


In [37]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'


root_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data'
batch_size = 2
num_workers = 0
transforms = None
dm = SonarSimDataModule(root_dir, batch_size, num_workers, transforms, train_config_pth)
dm.setup()
train_data_loader = dm.train_dataloader()

batch = next(iter(train_data_loader))
series, time, trajectory, depth = batch

print(series.shape)
print(time.shape)
print(trajectory.shape)
print(depth.shape)



torch.Size([2, 5, 1, 800, 768])
torch.Size([2, 5, 1])
torch.Size([2, 5, 7])
torch.Size([2, 5, 1])


In [38]:

def quat_norm(traj):
    x, y, z, w = traj[3], traj[4], traj[5], traj[6]
    norm = torch.sqrt(x**2 + y**2 + z**2 + w**2)
    return norm 

batch = next(iter(train_data_loader))
series, time, trajectory, depth = batch

# b_, n_, d_ = trajectory.shape
# for b in range(b_):
#     for n in range(n_):
#         single_traj = trajectory[b, n, :]
#         norm = quat_norm(single_traj)
#         print(f'for batch: {b}, frame: {n} norm: {norm}')

# **Training test**

In [39]:
dpso_t = DPSO(model_config_pth, sonar_config_pth, train_config_pth)
dpso_t.train()

DPSO_train(
  (PatchGraph): Graph(
    (patchifier): Patchifier(
      (feature_extractor): Encoder(
        (conv1): Conv2d(1, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (ResBlock1): Sequential(
          (0): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          )
          (1): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3

In [ ]:
from src.training.metrics import pose_err

optimizer = torch.optim.Adam(dpso_t.parameters(), lr=1e-4)

test_iter = 2
w1 = 1e-3
w2 = 1e5
w3 = 1e-3
w4 = 1e5

for i in range(test_iter):

    optimizer.zero_grad()

    loss = 0.0

    batch = next(iter(train_data_loader))
    series, time, trajectory, depth = batch

    pred = dpso_t(series, time, trajectory, depth, freeze_poses=False)

    for k, (pred_poses, proj_err) in enumerate(pred):

        trans_err, rot_err = pose_err(pred_poses, trajectory)
        proj_x_err = torch.mean(proj_err[:, 0], dim=-1)
        proj_y_err = torch.mean(proj_err[:, 1], dim=-1)

        # add weights: 
        
        loss += w1*trans_err + w2*rot_err + w3*proj_x_err + w4*proj_y_err
        # print(trans_err, rot_err, proj_x_err, proj_y_err)

        # print(f'    From update oper iter: {k}, loss components:\n    trans_err: {trans_err}\n    rot_err: {rot_err}\n    proj_x_err: {proj_x_err}\n    proj_y_err: {proj_y_err}')
    
    print(f'Iter: {i}, total loss: {loss}')

    loss.backward()
    optimizer.step()


Optim iter 0, valid edges: 21/180
[Warning] Cannot run BA for iteration 0: Jacobian contains Nan! Check your model and input!
Optim iter 1, valid edges: 180/180
[Warning] Cannot run BA for iteration 1: Jacobian contains Nan! Check your model and input!
Optim iter 2, valid edges: 180/180
[Warning] Cannot run BA for iteration 2: Jacobian contains Nan! Check your model and input!
Optim iter 3, valid edges: 180/180
[Warning] Cannot run BA for iteration 3: Jacobian contains Nan! Check your model and input!
Optim iter 4, valid edges: 180/180
[Warning] Cannot run BA for iteration 4: Jacobian contains Nan! Check your model and input!
Iter: 0, total loss: nan


In [41]:
# for n in range(pred_poses.shape[1]):
#     print(f'pred {n}:\n{pred_poses[0, n, :]}\n{trajectory[0, n, :]}')